<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b> Chat Memory Patterns</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Zustandslosigkeit von LLMs
---


Large Language Models (LLMs) wie GPT sind von Natur aus **zustandslos** - sie verfugen uber kein eingebautes Gedachtnis. Jede Anfrage wird isoliert verarbeitet, ohne Bezug zu vorherigen Interaktionen. Deshalb muss der Chatverlauf (Historie) bei jeder Anfrage neu ubergeben werden.

```
Ohne Memory:
User: "Mein Name ist Max"
AI: "Hallo Max!"
User: "Wie heisse ich?"
AI: "Das habe ich nicht gespeichert."
```

**Gängige Memory-Patterns:**

| Pattern | Beschreibung | Anwendungsfall |
|---------|--------------|----------------|
| **Python-Liste** | Einfachste Lösung | Prototyping, kurze Sessions |
| **Trimming** | Nur letzte N Nachrichten an LLM | Token-Limit einhalten |
| **Summary** | Alte Nachrichten zusammenfassen | Lange Sessions, Kontext erhalten |
| **SqliteSaver** | Persistente Speicherung | Production, Multi-User |

In [ ]:
#@markdown Chat Memory Patterns
diagram = """
%%{init: {'theme':'forest'}}%%
graph TD
    root[Chat Memory Patterns]

    %% 1. Manuelle Variante
    root -->|Short-term / Manuell| manual[Python-Liste]
    manual --> manual_desc["Einfache Liste im RAM<br/>Use Case: Prototyping, kurze Sessions"]

    %% 2. LangGraph Variante
    root -->|Automatisch / LangGraph| lg[StateGraph + InMemorySaver]

    %% Optimierungs-Strategien
    lg -->|Context Management| strat[Strategien mit Token-Limits]

    strat --> trim[Trimming]
    trim --> trim_desc["Nur letzte N Nachrichten an LLM<br/>State bleibt vollständig erhalten"]

    strat --> sum[Summary]
    sum --> sum_desc["RemoveMessage löscht alte Nachrichten<br/>Summary als SystemMessage im State"]

    %% Persistenz-Lösungen
    lg -->|persistentes Memory| persist[Persistenz-Backends]

    persist --> sqlite[SqliteSaver]
    sqlite --> sqlite_desc["SQLite-Datenbank<br/>Use Case: Persistenz, Multi-Thread"]

    %% Styling
    style root fill:#2ecc71,stroke:#27ae60,stroke-width:2px,color:white
    style lg fill:#3498db,stroke:#2980b9,stroke-width:2px,color:white
    style manual fill:#95a5a6,stroke:#7f8c8d,stroke-width:2px,color:white
    style strat fill:#f1c40f,stroke:#f39c12,color:black
    style persist fill:#e67e22,stroke:#d35400,color:white
"""
mermaid(diagram, width=800)


# 2 | LangGraph InMemorySaver
---

[LangGraph Explainer](https://editor.p5js.org/ralf.bendig.rb/full/EUzaFq4C4)


`LangGraph` ist ein Framework aus dem Umfeld von LangChain zur Entwicklung komplexer KI-Agenten und Workflows.
Es erweitert klassische LangChain-Ketten um ein *graphbasiertes* Zustandsmodell.

Die Grundidee:

Ein Workflow besteht aus Knoten (Nodes) und Übergängen (Kanten, Edges).
Jeder Knoten führt eine Aktion aus, z. B.:
+ LLM-Aufruf
+ Tool-Nutzung
+ Datenbankabfrage
+ Entscheidungslogik  


Der aktuelle Zustand wird während der Ausführung gespeichert und weitergegeben.

Dadurch eignet sich LangGraph besonders für:

+ Agentensysteme
+ Mehrschritt-Reasoning
+ Tool-Calling
+ Schleifen und Retry-Mechanismen
+ Multi-Agent-Architekturen
+ Zustandsbehaftete Chatbots

Bei einfachen Chatbots lässt sich der bisherige Gesprächsverlauf noch manuell in einer Liste speichern. Bei größeren und komplexeren KI-Anwendungen wird diese manuelle Verwaltung jedoch schnell unübersichtlich. Ohne ein strukturiertes System vergisst eine KI nach jedem Schritt den Kontext des vorherigen Gesprächs.

**Lösung: Automatische Zustandsverwaltung mit LangGraph**

*LangGraph* übernimmt die Verwaltung des Chat-Verlaufs vollautomatisch. Das Prinzip lässt sich mit der *Auto-Save-Funktion eines Software-Systems* vergleichen:

* **Der Zustand (State):** Über einen vorbereiteten Zustandstyp namens `MessagesState` wird ein digitaler Notizblock bereitgestellt. Auf diesem wird jede Benutzeranfrage und jede KI-Antwort automatisch dokumentiert.
* **Der Speicher-Mechanismus (Checkpointer):** Nach jedem einzelnen Verarbeitungsschritt sichert ein sogenannter Checkpointer den aktuellen Zustand. Bei der nächsten Interaktion wird genau dieser Zustand wieder geladen, sodass nahtlos an das vorherige Gespräch angeknüpft werden kann.



**Die Speicheroptionen:**

Je nach Anforderung kann definiert werden, wo diese Daten gesichert werden:

1. **InMemorySaver (Flüchtiger Speicher):** Die Chats werden im Arbeitsspeicher abgelegt. Nach einem Neustart des Programms ist der Verlauf gelöscht.
2. **SqliteSaver (Dauerhafter Speicher):** Die Daten werden in einer SQLite-Datenbankdatei auf der Festplatte gesichert. Der Verlauf bleibt somit auch nach einem Neustart des Systems erhalten.



**Funktionsweise im Hintergrund: Anhängen statt Überschreiben**

In der Programmierung werden Variablen bei neuen Daten häufig überschrieben. LangGraph nutzt stattdessen ein Additions-Prinzip über die interne Funktion `add_messages`. Neue Antworten werden immer an das bestehende Protokoll angehängt.



**Ein Beispiel:**

> 1. **Ausgangslage im Speicher:**
> `[ HumanMessage("Hallo"), AIMessage("Hi!") ]`
> 2. **Generierung einer neuen Antwort:**
> `AIMessage("Wie kann ich helfen?")`
> 3. **Aktualisierter Zustand:**
> LangGraph ersetzt die alten Nachrichten nicht, sondern ergänzt die Liste:     
> `[ HumanMessage("Hallo"), AIMessage("Hi!"), AIMessage("Wie kann ich helfen?") ]`



**Der entscheidende Vorteil:** Es ist kein manueller Code notwendig, um die Liste zu aktualisieren oder alte Nachrichten vor dem Überschreiben zu schützen. Das System steuert den gesamten Ablauf im Hintergrund.

<p><font color='black' size="5">
2.1 | Basismodell
</font></p>

---

In [ ]:
# Import

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_core.messages import SystemMessage, HumanMessage

# ── LangGraph ─────────────────────────────────────────────────────────────────
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState, START

In [ ]:
def create_memory_app(model_fn, checkpointer):
    """Erstellt einen linearen StateGraph mit einem Model-Node."""
    g = StateGraph(MessagesState)
    g.add_node("model", model_fn)
    g.add_edge(START, "model")
    return g.compile(checkpointer=checkpointer)

print("Graph-Factory bereit ✅")

In [ ]:
#@markdown Funktionsaufrufe Basismodell
diagram = """
flowchart LR
    %% Definition der Stile für Knoten
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef graphStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;

    %% 1. Bereich: Benutzereingabe / Schleife (Blau)
    subgraph UI ["<b>1. Input / Output (Code-Funktionen)</b>"]
        direction TB
        Start["`<b>ask(thread_id, user_input)</b><br>Startet Chat via <i>app.invoke()</i>`"]:::loopStyle
        Ende["`<b>mprint() / Ausgabe</b><br>Gibt <i>response</i> im Terminal aus`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    %% 2. Bereich: LangGraph Kernprozess (Orange)
    subgraph Engine ["<b>2. LangGraph-Zustandsmaschine (app)</b>"]
        direction TB
        Load["`<b>1. State laden</b><br>Holt <i>MessagesState</i> für thread_id`"]:::graphStyle
        LLM["`<b>2. Node: 'model' (call_model)</b><br>Verarbeitet <i>SystemMessage + state['messages']</i>`"]:::graphStyle
        Save["`<b>3. State speichern</b><br>Hängt <i>AIMessage</i> via Reducer an`"]:::graphStyle

        Load --> LLM --> Save
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    %% 3. Bereich: Der Speicher (Lila)
    subgraph Storage ["<b>3. Checkpointer (memory)</b>"]
        Memory["`<b>memory = InMemorySaver()</b><br>Speichert Zustände im RAM`"]:::saveStyle
    end
    style Storage fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    %% Verbindungen zwischen den Bereichen
    Start --> Load
    Load <--> Memory
    Save --> Memory
    Save --> Ende
"""
mermaid(diagram, width=1100)


<p><font color='violet' size="4">
3. Checkpointer (memory)
</font></p>

<p><font color='orange' size="4">
2. LangGraph-Zustandsmaschine (app)
</font></p>

In [ ]:
# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain.chat_models import init_chat_model

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.model_config import BASELINE

# LLM: konfiguriertes Sprachmodell
llm = init_chat_model(BASELINE)

# System-Prompt: Grundanweisung an das Modell
system_prompt = "Du bist ein hilfreicher und humorvoller KI-Assistent."

print(f"LLM: {BASELINE} ✅")

In [ ]:
# Node-Definition: Bereitet Nachrichten vor und ruft das LLM auf
def call_model(state: MessagesState):
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    return {"messages": [llm.invoke(messages)]}

In [ ]:
app = create_memory_app(call_model, InMemorySaver())
print("StateGraph mit InMemorySaver definiert ✅")

In [ ]:
# Hilfsfunktion: Expliziten Zustand (State) auslesen
def get_thread_history(thread_id: str) -> list:
    """Gibt die gespeicherten Nachrichten eines Threads zurück."""
    config = {"configurable": {"thread_id": thread_id}}
    state = app.get_state(config)
    return state.values.get("messages", [])

<p><font color='blue' size="4">
1. Input/Output
</font></p>

In [ ]:
def ask(thread_id: str, user_input: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)
    response = result["messages"][-1].content
    mprint(f"**[{thread_id}] Mensch:** {user_input}")
    mprint(f"**[{thread_id}] KI:** {response}\n")
    return response

In [ ]:
mprint("## InMemorySaver Demo")
mprint("---")

for thread_id, user_input in [
    ("max", "Hallo! Ich bin Max aus München."),
    ("max", "Ich programmiere gerne in Python."),
    ("emma", "Hi! Ich bin Emma und mag Machine Learning."),
    ("max", "Woher komme ich und was ist mein Hobby?"),
]:
    ask(thread_id, user_input)

mprint("### Gespeicherte Threads")
for thread_id in ["max", "emma"]:
    history = get_thread_history(thread_id)
    mprint(f"- **{thread_id}**: {len(history)} Nachrichten")

mprint("### Historie: max")
for index, message in enumerate(get_thread_history("max"), 1):
    role = "Mensch" if message.type == "human" else "KI"
    mprint(f"{index}. **{role}:** {message.content}")


<p><font color='black' size="5">
2.2 | Trimming (Sliding Window)
</font></p>

---

**Problem:** Bei langen Konversationen wächst `state["messages"]` unbegrenzt — das Token-Limit des LLM wird überschritten.

**Lösung:** `trim_messages()` im `call_model`-Node begrenzt den an das LLM übergebenen Kontext. Der State bleibt vollständig erhalten — nur das LLM-Kontextfenster wird begrenzt.

> Um auch den State physisch zu kürzen, können `RemoveMessage`-Operationen eingesetzt werden (Abschnitt 2.3).

In [ ]:
#@markdown Funktionsaufrufe Trimming
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef graphStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;

    subgraph UI ["<b>1. Input / Output (Code-Funktionen)</b>"]
        direction TB
        Start["`<b>app_trimmed.invoke()</b><br>Startet Chat mit thread_id`"]:::loopStyle
        Ende["`<b>mprint() / Ausgabe</b><br>Gibt response im Terminal aus`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["<b>2. LangGraph-Zustandsmaschine (app_trimmed)</b>"]
        direction TB
        Load["`<b>1. State laden</b><br>Vollständige messages-Liste`"]:::graphStyle
        Trim["`<b>2. trim_messages()</b><br>Letzte MAX_MESSAGES Nachrichten`"]:::graphStyle
        LLM["`<b>3. Node: call_model_trimmed</b><br>SystemMsg + trimmed -> llm.invoke()`"]:::graphStyle
        Save["`<b>4. State speichern</b><br>Vollständige Historie bleibt erhalten`"]:::graphStyle
        Load --> Trim --> LLM --> Save
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph TrimCfg ["<b>3. Trimmer-Konfiguration</b>"]
        direction TB
        Config["`<b>trimmer = trim_messages()</b><br>max_tokens=MAX_MESSAGES=6<br>strategy=last`"]:::saveStyle
        Memory["`<b>InMemorySaver()</b><br>Speichert vollständigen State`"]:::saveStyle
    end
    style TrimCfg fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Load <--> Memory
    Trim -.->|nutzt| Config
    Save --> Memory
    Save --> Ende
"""
mermaid(diagram, width=1100)


<p><font color='violet' size="4">
3. Trimmer-Konfiguration
</font></p>

In [ ]:
# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_core.messages import trim_messages

MAX_MESSAGES = 6

trimmer = trim_messages(
    max_tokens=MAX_MESSAGES,
    strategy="last",
    token_counter=len,
)

<p><font color='orange' size="4">
2. LangGraph-Zustandsmaschine (app_trimmed)
</font></p>

In [ ]:
def call_model_trimmed(state: MessagesState):
    trimmed = trimmer.invoke(state["messages"])
    response = llm.invoke([SystemMessage(content=system_prompt)] + trimmed)
    return {"messages": [response]}

In [ ]:
app_trimmed = create_memory_app(call_model_trimmed, InMemorySaver())
print(f"Graph mit Trimming erstellt (max. {MAX_MESSAGES} Nachrichten im LLM-Kontext) ✅")

<p><font color='blue' size="4">
1. Input/Output
</font></p>

In [ ]:
mprint("## Trimming Demo (max. 6 Nachrichten im LLM-Kontext)")
mprint("---")

config_trim = {"configurable": {"thread_id": "trim_test"}}

for nr, user_input in enumerate([
    "Mein Geheimcode lautet: BLAU-42.",   # wird vom LLM nicht wiederholt → echter Test
    "Ich komme aus München.",
    "Ich mag Python.",
    "Ich arbeite als Data Scientist.",
    "Was war mein Geheimcode?",           # fällt aus dem Trimming-Fenster
], 1):
    result = app_trimmed.invoke({"messages": [HumanMessage(content=user_input)]}, config=config_trim)
    response = result["messages"][-1].content
    mprint(f"**{nr}. Mensch:** {user_input}")
    mprint(f"**{nr}. KI:** {response}\n")

<p><font color='black' size="4">
Zustandsgraph
</font></p>

In [ ]:
# State enthält alle Nachrichten — Trimming betrifft nur das LLM-Kontextfenster
state = app_trimmed.get_state(config_trim)
all_messages = state.values.get("messages", [])
trimmed_window = trimmer.invoke(all_messages)
cutoff = len(all_messages) - len(trimmed_window)

mprint("### Vollständige Historie im State")
mprint(f"{len(all_messages)} Nachrichten gespeichert · LLM sah zuletzt {len(trimmed_window)}")
mprint("---")

for index, message in enumerate(all_messages, 1):
    role = "Mensch" if message.type == "human" else "KI"
    marker = "✅" if index > cutoff else "❌"
    mprint(f"{marker} {index}. **{role}:** {message.content}")

mprint("---")
mprint(f"\n**Legende:** ✅ im Trimming-Fenster (LLM sah diese) · ❌ entfernt (zu alt)")
mprint(f"\n**Ergebnis:** Das Modell sah maximal {MAX_MESSAGES} Nachrichten — der State enthält alle {len(all_messages)}.")
mprint(f"👆 'BLAU-42' steckt nur in Nachricht ❌ 1 — das LLM kann ihn in Runde 5 nicht mehr kennen.")


<p><font color='black' size="5">
2.3 | Summary (LLM-basierte Zusammenfassung)
</font></p>

---

**Problem:** Trimming verwirft alte Informationen vollständig — wichtiger Kontext geht verloren.

**Alternative:** **Summarization** — Ein LLM fasst alte Nachrichten zusammen, die Zusammenfassung ersetzt sie im State.

**Strategie:**
1. Wenn `state["messages"]` > Limit: Alte Nachrichten zusammenfassen
2. `RemoveMessage` löscht alte Nachrichten physisch aus dem State
3. Summary als `SystemMessage` + letzte N Nachrichten bleiben erhalten

```python
# Pseudo-Code
if len(state["messages"]) > SUMMARY_THRESHOLD:
    summary = llm.invoke("Fasse zusammen: {old_messages}")
    return {
        "messages": [RemoveMessage(id=m.id) for m in old_messages]
                  + [SystemMessage(summary), llm_response]
    }
```

In [ ]:
#@markdown Funktionsaufrufe Summary
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef graphStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;
    classDef decStyle fill:#fce4ec,stroke:#e91e63,stroke-width:2px;

    subgraph UI ["<b>1. Input / Output (Code-Funktionen)</b>"]
        direction TB
        Start["`<b>app_summary.invoke()</b><br>Startet Chat mit thread_id`"]:::loopStyle
        Ende["`<b>mprint() / Ausgabe</b><br>Gibt response im Terminal aus`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["<b>2. LangGraph-Zustandsmaschine (app_summary)</b>"]
        direction TB
        Load["`<b>1. State laden</b><br>messages-Liste`"]:::graphStyle
        Check{"`<b>len > THRESHOLD?</b>`"}:::decStyle
        Sum["`<b>2a. summarize_old_messages()</b><br>LLM fasst alte Nachrichten zusammen`"]:::graphStyle
        Remove["`<b>2b. RemoveMessage()</b><br>Alte Nachrichten aus State löschen`"]:::graphStyle
        LLM["`<b>3. Node: call_model_summary</b><br>context + llm.invoke()`"]:::graphStyle
        Save["`<b>4. State speichern</b><br>Summary + letzte N Nachrichten`"]:::graphStyle
        Load --> Check
        Check -->|Ja| Sum --> Remove --> LLM --> Save
        Check -->|Nein| LLM
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph SumCfg ["<b>3. Summary-Konfiguration</b>"]
        direction TB
        Cfg["`<b>SUMMARY_THRESHOLD = 8</b><br>KEEP_RECENT = 4`"]:::saveStyle
        Memory["`<b>InMemorySaver()</b><br>Speichert komprimierten State`"]:::saveStyle
    end
    style SumCfg fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Load <--> Memory
    Check -.->|nutzt| Cfg
    Save --> Memory
    Save --> Ende
"""
mermaid(diagram, width=1100)


<p><font color='violet' size="4">
3. Summary-Konfiguration
</font></p>

In [ ]:
# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_core.messages import RemoveMessage
# RemoveMessage(id=m.id) markiert eine Nachricht zum physischen Löschen aus dem State.
# Jede LangGraph-Nachricht erhält beim Erstellen automatisch eine eindeutige id.

SUMMARY_THRESHOLD = 8
KEEP_RECENT = 4

<p><font color='orange' size="4">
2. LangGraph-Zustandsmaschine (app_summary)
</font></p>

In [ ]:
def summarize_old_messages(old_messages: list) -> str:
    prompt = (
        "Extrahiere alle genannten Fakten aus der Konversation als strukturierte Liste.\n"
        "Format: 'Name: ..., Alter: ..., Herkunft: ..., Beruf: ..., Hobbys: ..., Vorlieben: ...'\n"
        "Nur belegte Werte aufführen — fehlende Felder weglassen."
    )
    return llm.invoke([SystemMessage(content=prompt)] + old_messages).content

In [ ]:
def _compress_messages(messages: list):
    """Fasst alte Nachrichten zusammen und erstellt RemoveMessage-Operationen."""
    to_summarize = messages[:-KEEP_RECENT]
    summary_text = summarize_old_messages(to_summarize)
    summary_msg  = SystemMessage(content=f"Zusammenfassung bisher:\n{summary_text}")
    remove_ops   = [RemoveMessage(id=m.id) for m in to_summarize]
    return remove_ops, summary_msg

In [ ]:
def call_model_summary(state: MessagesState):
    messages = state["messages"]

    if len(messages) > SUMMARY_THRESHOLD:
        remove_ops, summary_msg = _compress_messages(messages)
        context  = [SystemMessage(content=system_prompt), summary_msg] + list(messages[-KEEP_RECENT:])
        response = llm.invoke(context)
        return {"messages": remove_ops + [summary_msg, response]}

    # Kein Threshold: SystemMessages (Zusammenfassungen) immer vor Chat-Nachrichten
    summary_msgs = [m for m in messages if isinstance(m, SystemMessage)]
    conv_msgs    = [m for m in messages if not isinstance(m, SystemMessage)]
    response     = llm.invoke([SystemMessage(content=system_prompt)] + summary_msgs + conv_msgs)
    return {"messages": [response]}

In [ ]:
app_summary = create_memory_app(call_model_summary, InMemorySaver())
print(f"Graph mit Summary erstellt (Limit: {SUMMARY_THRESHOLD}, behalte letzte {KEEP_RECENT}) ✅")

<p><font color='blue' size="4">
1. Input/Output
</font></p>

In [ ]:
mprint("## Summary Demo (Limit: 8 Nachrichten, behalte 4)")
mprint("---")

config_summary = {"configurable": {"thread_id": "summary_test"}}

for i, user_msg in enumerate([
    "Mein Name ist Max.",
    "Ich bin 30 Jahre alt.",
    "Ich komme aus München.",
    "Ich arbeite als Data Scientist.",
    "Mein Hobby ist Fotografie.",
    "Ich mag italienisches Essen.",], 1):

    result = app_summary.invoke({"messages": [HumanMessage(content=user_msg)]}, config=config_summary)
    mprint(f"**{i}. Mensch:** {user_msg}")
    mprint(f"**{i}. KI:** {result['messages'][-1].content}\n")

In [ ]:
# Erinnerungstest
result = app_summary.invoke(
    {"messages": [HumanMessage(content="Fasse zusammen: Wie heisse ich, wie alt bin ich, woher komme ich?")]},
    config=config_summary
)
mprint(f"**Test. Mensch:** Fasse zusammen: Wie heisse ich, wie alt bin ich, woher komme ich?")
mprint(f"**Test. KI:** {result['messages'][-1].content}")

state = app_summary.get_state(config_summary)
msgs_in_state = state.values.get("messages", [])
mprint(f"\n**State nach Summary:** {len(msgs_in_state)} Nachrichten (komprimiert)")
mprint("✅ **Ergebnis:** Der Bot erinnert sich via Summary an alte Infos!")

# 3 | Persistentes Memory
---


<p><font color='black' size="5">
3.1 | SqliteSaver (Persistente Speicherung)
</font></p>

---


**Problem:** `InMemorySaver` verliert alle Daten beim Neustart der Laufzeitumgebung (Colab-Session).

**Lösung:** **SqliteSaver** — speichert den Graph-State persistent in einer SQLite-Datenbank. Der Austausch ist minimal:

```python
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# Verbindung öffnen (check_same_thread=False für Notebook-Kompatibilität)
conn = sqlite3.connect("./sessions.db", check_same_thread=False)
app = workflow.compile(checkpointer=SqliteSaver(conn))
```

**Vorteile:**
- Daten bleiben nach Neustart erhalten
- Kein eigener Datenbankcode nötig
- Thread-Verwaltung übernimmt der Checkpointer


In [ ]:
#@markdown Funktionsaufrufe SqliteSaver
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef graphStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;

    subgraph UI ["<b>1. Input / Output (Code-Funktionen)</b>"]
        direction TB
        Start["`<b>app_persistent.invoke()</b><br>Startet Chat mit thread_id`"]:::loopStyle
        Ende["`<b>mprint() / Ausgabe</b><br>Gibt response im Terminal aus`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["<b>2. LangGraph-Zustandsmaschine (app_persistent)</b>"]
        direction TB
        Load["`<b>1. State laden</b><br>MessagesState für thread_id`"]:::graphStyle
        LLM["`<b>2. Node: call_model</b><br>SystemMessage + state['messages'] -> LLM`"]:::graphStyle
        Save["`<b>3. State speichern</b><br>AIMessage via Reducer angehängt`"]:::graphStyle
        Load --> LLM --> Save
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph Storage ["<b>3. Checkpointer (SqliteSaver)</b>"]
        direction TB
        Conn["`<b>sqlite3.connect()</b><br>DB-Verbindung öffnen<br>check_same_thread=False`"]:::saveStyle
        Saver["`<b>SqliteSaver(conn)</b><br>Persistenter Checkpointer<br>Speichert State in SQLite-Datei`"]:::saveStyle
        Conn --> Saver
    end
    style Storage fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Load <--> Saver
    Save --> Saver
    Save --> Ende
"""
mermaid(diagram, width=1100)


<p><font color='violet' size="4">
3. Checkpointer (SqliteSaver)
</font></p>

In [ ]:
# ── LangGraph ─────────────────────────────────────────────────────────────────
from langgraph.checkpoint.sqlite import SqliteSaver
print("SqliteSaver verfügbar ✅")

In [ ]:
# ── Datenbanken ───────────────────────────────────────────────────────────────
import os
import sqlite3

DB_PATH = "./chat_memory.db"

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
print(f"DB-Verbindung geöffnet: {DB_PATH}")

<p><font color='orange' size="4">
2. LangGraph-Zustandsmaschine (app_persistent)
</font></p>

In [ ]:
app_sqlite = create_memory_app(call_model, SqliteSaver(conn))
print(f"Graph mit SqliteSaver kompiliert ✅")
print(f"Speicherort: {DB_PATH}")

<p><font color='blue' size="4">
1. Input/Output
</font></p>

In [ ]:
mprint("## SqliteSaver Demo")
mprint("---")

config_persist = {"configurable": {"thread_id": "max_persistent"}}

for msg in [
    "Hallo! Ich bin Max aus München.",
    "Ich mag Python-Programmierung.",
    "Woher komme ich und was mag ich?",
]:
    result = app_sqlite.invoke({"messages": [HumanMessage(content=msg)]}, config=config_persist)
    mprint(f"**Mensch:** {msg}")
    mprint(f"**KI:** {result['messages'][-1].content}\n")

In [ ]:
state = app_sqlite.get_state(config_persist)
messages = state.values.get("messages", [])

mprint("### Gespeicherter Thread:")
mprint(f"- **max_persistent**: {len(messages)} Nachrichten in `{DB_PATH}`")

In [ ]:
mprint("### Test: Persistenz-Check")
mprint("---")

# Neue Verbindung zur gleichen Datei (simuliert Neustart)
conn_restart = sqlite3.connect(DB_PATH, check_same_thread=False)
app_after_restart = create_memory_app(call_model, SqliteSaver(conn_restart))

config_restart = {"configurable": {"thread_id": "max_persistent"}}
result = app_after_restart.invoke(
    {"messages": [HumanMessage(content="Was haben wir bisher besprochen?")]},
    config=config_restart
)
mprint(f"**Mensch (nach 'Neustart'):** Was haben wir bisher besprochen?")
mprint(f"**KI:** {result['messages'][-1].content}")
mprint("\n✅ **Ergebnis:** Die Historie überlebt den Neustart!")

In [ ]:
def chat_sqlite(thread_id: str, user_input: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app_sqlite.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)
    return result["messages"][-1].content

def show_history(thread_id: str):
    config   = {"configurable": {"thread_id": thread_id}}
    state    = app_sqlite.get_state(config)
    messages = state.values.get("messages", [])
    mprint(f"### Thread: {thread_id} ({len(messages)} Nachrichten)")
    mprint("---")
    for i, msg in enumerate(messages, 1):
        role = "Mensch" if msg.type == "human" else "KI"
        mprint(f"{i}. **{role}:** {msg.content}")

In [ ]:
mprint("## Multi-Thread Demo")
mprint("---")

mprint("### Thread: max_session")
mprint(f"**KI:** {chat_sqlite('max_session', 'Hallo! Ich bin Max und komme aus München.')}")
mprint(f"**KI:** {chat_sqlite('max_session', 'Ich interessiere mich für Machine Learning.')}")

mprint("\n### Thread: emma_session")
mprint(f"**KI:** {chat_sqlite('emma_session', 'Hi! Ich bin Emma aus Berlin.')}")

mprint("\n### Zurück zu max_session")
mprint(f"**KI:** {chat_sqlite('max_session', 'Woher komme ich nochmal?')}")

mprint("\n### Alle Threads:")
mprint("---")
for thread_id in ["max_session", "emma_session"]:
    config = {"configurable": {"thread_id": thread_id}}
    state  = app_sqlite.get_state(config)
    count  = len(state.values.get("messages", []))
    mprint(f"- **{thread_id}**: {count} Nachrichten")

show_history("max_session")

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> Wichtig ist nur: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**

Teste **Trimming**-Limits in einem Chatgespräch und beobachte, ab wann der Kontext verloren geht (mind. 5 Nachrichten). Der Chatverlauf kann entweder als Python-Liste oder mit einem StateGraph umgesetzt werden.

**✅ Erledigt wenn:** Ab einer bestimmten Nachrichtenanzahl gibt das Modell an, die frühere Frage nicht mehr zu kennen — der Grenzwert ist dokumentiert.

In [ ]:
# Grundlagen: Trimming-Limits testen
# Startpunkt: Trimming-Beispiel aus Kapitel 2 als Vorlage

# Ablauf: Chatgespräch mit mindestens 5 Nachrichten führen
# Nach jeder Runde fragen: 'Was war meine erste Frage?'
# Beobachten: Ab wann ist die Antwort 'Das weiß ich nicht mehr'
# Grenzwert dokumentieren
# ...

**Aufbau**

Verbessere den Summary-Prompt oder baue einen interaktiven CLI-Chatbot mit Threads, der Kontext über mehrere Nachrichten hält. Der Chatverlauf kann entweder als Python-Liste oder mit einem StateGraph umgesetzt werden.

**✅ Erledigt wenn:** Der Chatbot hält Kontext über mindestens fünf aufeinander aufbauende Fragen — eine Rückfrage auf die erste Frage wird korrekt beantwortet.

In [ ]:
# Aufbau: CLI-Chatbot mit Summary oder Threads
# Startpunkt: Memory-Beispiele aus Kapitel 3/4 als Vorlage

# Option A: Summary-Memory — Zusammenfassung statt vollem Verlauf
# Option B: Interaktiver CLI-Chatbot mit while-Schleife und Thread-ID

# Test: 5 aufeinander aufbauende Fragen stellen
# Dann: 'Was war meine erste Frage?' → muss korrekt beantwortet werden
# ...

**Vertiefung**

Kombiniere Trimming, Summary und SQLite zu einem Hybrid-Memory-System für persistente Chatsitzungen.

**✅ Erledigt wenn:** Der Chatbot liest beim Start gespeicherte Sitzungen aus der SQLite-Datenbank — Kontext aus früheren Sitzungen fließt in die Antwort ein.

In [ ]:
# Vertiefung: Hybrid-Memory-System (Trimming + Summary + SQLite)
# Startpunkt: SQLite-Beispiele aus Abschnitt B dieses Notebooks

# 1. Trimming: kurzen Kontext im Arbeitsspeicher halten
# 2. Summary: ältere Nachrichten zusammenfassen und speichern
# 3. SQLite: Sitzungen persistent speichern und beim Start laden

# Test: Colab neu starten → gespeicherter Kontext muss wiederhergestellt werden
# ...

# B | Datenbank auslesen
---

Threads aus `app_sqlite` auslesen — nützlich für Debugging, Analyse oder Export.

In [ ]:
def read_all_threads(thread_ids: list, app_instance=None):
    """Liest alle Threads via LangGraph State API."""
    if app_instance is None:
        app_instance = app_sqlite

    mprint(f"### Alle Threads ({len(thread_ids)} gesamt)")
    mprint("---")

    for thread_id in thread_ids:
        config = {"configurable": {"thread_id": thread_id}}
        state = app_instance.get_state(config)
        messages = state.values.get("messages", [])

        if not messages:
            continue

        mprint(f"#### Thread: {thread_id} ({len(messages)} Nachrichten)")
        for i, msg in enumerate(messages, 1):
            role = "Mensch" if msg.type == "human" else "KI"
            content_short = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
            mprint(f"{i}. **{role}:** {content_short}")
        mprint("")

In [ ]:
read_all_threads(["max_session", "emma_session"])

In [ ]:
def delete_db_files():
    """Löscht die SQLite-Datenbank (Cleanup)."""
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Gelöscht: {DB_PATH}")
    else:
        print(f"Nicht gefunden: {DB_PATH}")

# Auskommentiert, um versehentliches Löschen zu verhindern:
# delete_db_files()

# C | Dokumente zum Weiterlesen
---



📚 Ergänzende Artikel aus der Kurs-Dokumentation:

- [Memory-Systeme](https://ralf-42.github.io/GenAI/03-grundlagen/memory-systeme.html)
- [Context Engineering](https://ralf-42.github.io/GenAI/05-prompting-rag/context-engineering.html)
- [Einsteiger LangGraph](https://ralf-42.github.io/GenAI/06-frameworks/einsteiger-langgraph.html)
- [LangGraph Best Practices](https://ralf-42.github.io/GenAI/06-frameworks/langgraph-best-practices.html)